# AI4I Benchmark Reconstruction

This notebook is the public entry point for reconstructing the AI4I benchmark used in the paper.

## Inputs
- Expected raw dataset path: `../dataset/ai4i2020.csv`

## Outputs
- Stage-wise split manifests
- Stage-wise split summaries
- Prototypical knowledge-base JSON files
- Placeholder supplementary JSON files
- Observed test-case graph JSON files

## Notes
- This public notebook is intentionally minimal.
- Helper logic is loaded from `../src/ai4i_data_generation_utils.py`.
- Place the raw AI4I CSV file before running all cells.

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = Path('..').resolve()
SRC_ROOT = ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import ai4i_data_generation_utils

ai4i_data_generation_utils = importlib.reload(ai4i_data_generation_utils)
build_dataset_bundle = ai4i_data_generation_utils.build_dataset_bundle
make_default_stage_configs = ai4i_data_generation_utils.make_default_stage_configs

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

DATASET_PATH = ROOT / 'dataset' / 'ai4i2020.csv'
OUTPUT_ROOT = ROOT / 'outputs' / 'ai4i_benchmark'

assert DATASET_PATH.exists(), f'Missing dataset: {DATASET_PATH}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def to_rel(path: str | Path) -> str:
    try:
        return Path(path).resolve().relative_to(ROOT).as_posix()
    except ValueError:
        return Path(path).as_posix()

print('Dataset:', to_rel(DATASET_PATH))
print('Output root:', to_rel(OUTPUT_ROOT))
print('Source root:', to_rel(SRC_ROOT))

In [ ]:
stage_configs = make_default_stage_configs()
stage_table = pd.DataFrame(
    [
        {
            'stage_name': cfg.stage_name,
            'reference_graph_policy': cfg.reference_graph_policy,
            'hybrid_alpha': cfg.hybrid_alpha,
            'split_strategy': cfg.split.strategy,
            'description': cfg.description,
        }
        for cfg in stage_configs.values()
    ]
).sort_values(['split_strategy', 'stage_name']).reset_index(drop=True)
stage_table

In [ ]:
TARGET_STAGES = [
    'rule_based',
    'hybrid_0.2',
    'hybrid_0.4',
    'hybrid_0.6',
    'hybrid_0.8',
    'empirical',
    'empirical_iid',
]

results = {}
for stage_name in TARGET_STAGES:
    results[stage_name] = build_dataset_bundle(
        dataset_path=DATASET_PATH,
        config=stage_configs[stage_name],
        output_root=OUTPUT_ROOT,
        relative_base=ROOT,
    )

summary_df = pd.DataFrame([results[name]['summary'] for name in TARGET_STAGES])
summary_df[[
    'stage_name',
    'reference_graph_policy',
    'hybrid_alpha',
    'split_strategy',
    'output_root',
]]

In [ ]:
empirical_head = results['empirical']['test_df'][['UDI', 'benchmark_mode', 'ambiguity_score']].head(10).assign(stage_name='empirical')
empirical_iid_head = results['empirical_iid']['test_df'][['UDI', 'benchmark_mode', 'ambiguity_score']].head(10).assign(stage_name='empirical_iid')

split_compare = pd.DataFrame(
    [
        {
            'stage_name': stage_name,
            'split_strategy': results[stage_name]['summary']['split_strategy'],
            'test_rows': len(results[stage_name]['test_df']),
            'test_ambiguity_mean': float(results[stage_name]['test_df']['ambiguity_score'].mean()),
            'test_ambiguity_median': float(results[stage_name]['test_df']['ambiguity_score'].median()),
        }
        for stage_name in ['empirical', 'empirical_iid']
    ]
)

display(split_compare)
pd.concat([empirical_head, empirical_iid_head], ignore_index=True)